In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("../Datos/datos_limpios.csv")

df.head()

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
# Copiamos el dataframe para no dañar el original
df_modelo = df.copy()

# Variable objetivo
y = df_modelo["nivel_cobertura"]

# Variables predictoras
X = df_modelo.drop(columns=["nivel_cobertura"])

In [ ]:
# Convertir columnas de texto a números
label_encoders = {}

for columna in X.select_dtypes(include=["object"]).columns:
    le = LabelEncoder()
    X[columna] = le.fit_transform(X[columna].astype(str))
    label_encoders[columna] = le

# Convertir variable objetivo si está en texto
le_y = LabelEncoder()
y = le_y.fit_transform(y.astype(str))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    random_state=42,
    stratify=y
)

In [ ]:
modelo_rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    max_depth=10,
    class_weight="balanced"
)

modelo_rf.fit(X_train, y_train)

In [ ]:
y_pred = modelo_rf.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print("Exactitud del modelo Random Forest:", round(accuracy, 4))
print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred, target_names=le_y.classes_))

In [ ]:
matriz = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(
    matriz, 
    annot=True, 
    fmt="d", 
    cmap="Blues",
    xticklabels=le_y.classes_,
    yticklabels=le_y.classes_
)

plt.title("Matriz de Confusión - Random Forest")
plt.xlabel("Predicción")
plt.ylabel("Valor Real")
plt.show()

In [ ]:
importancias = pd.DataFrame({
    "Variable": X.columns,
    "Importancia": modelo_rf.feature_importances_
})

importancias = importancias.sort_values(by="Importancia", ascending=False)

importancias.head(15)

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=importancias.head(15),
    x="Importancia",
    y="Variable"
)

plt.title("Variables más importantes en el modelo Random Forest")
plt.xlabel("Importancia")
plt.ylabel("Variable")
plt.show()

In [ ]:
import joblib

joblib.dump(modelo_rf, "../Modelos_entrenados/random_forest.pkl")

print("Modelo Random Forest guardado correctamente.")